# Advanced: Multi-Strategy Email Generation with PAMOLA.CORE

**Goal**: Master all 3 email generation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Name-Surname format with domain preservation
- **Strategy 2**: Surname-Name format with business domains
- **Strategy 3**: Nickname format with random domains
- Calculate advanced privacy metrics
- Analyze email format and domain distribution
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of email privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── fake_data/email/
        └── 02_fake_email_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.fake_data.operations.email_op import FakeEmailOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 6 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic names
    first_names = [
        'John', 'Jane', 'Robert', 'Mary', 'Michael', 'Patricia', 'David', 'Jennifer',
        'William', 'Linda', 'Richard', 'Barbara', 'Joseph', 'Susan', 'Thomas', 'Jessica',
        'Charles', 'Sarah', 'Christopher', 'Karen', 'Daniel', 'Nancy', 'Matthew', 'Lisa',
        'Anthony', 'Betty', 'Mark', 'Margaret', 'Donald', 'Sandra', 'Steven', 'Ashley',
        'Paul', 'Kimberly', 'Andrew', 'Emily', 'Joshua', 'Donna', 'Kenneth', 'Michelle',
        'Kevin', 'Carol', 'Brian', 'Amanda', 'George', 'Dorothy', 'Timothy', 'Melissa',
        'Ronald', 'Deborah', 'Edward', 'Stephanie', 'Jason', 'Rebecca', 'Jeffrey', 'Sharon'
    ]
    
    last_names = [
        'Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
        'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson', 'Thomas',
        'Taylor', 'Moore', 'Jackson', 'Martin', 'Lee', 'Perez', 'Thompson', 'White',
        'Harris', 'Sanchez', 'Clark', 'Ramirez', 'Lewis', 'Robinson', 'Walker', 'Young',
        'Allen', 'King', 'Wright', 'Scott', 'Torres', 'Nguyen', 'Hill', 'Flores',
        'Green', 'Adams', 'Nelson', 'Baker', 'Hall', 'Rivera', 'Campbell', 'Mitchell'
    ]
    
    departments = ['Engineering', 'Sales', 'Marketing', 'HR', 'Finance', 'Operations', 'Research']
    
    # Generate dataset
    df = pd.DataFrame({
        'employee_id': range(1, n_records + 1),
        'first_name': np.random.choice(first_names, n_records),
        'last_name': np.random.choice(last_names, n_records),
        'department': np.random.choice(departments, n_records),
        'hire_year': np.random.randint(2015, 2024, n_records),
        'email': [f"{np.random.choice(first_names).lower()}.{np.random.choice(last_names).lower()}@company.com" 
                  for _ in range(n_records)]
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Domain Dictionary (Optional)

**How to use:**
- Run to check/create custom domain dictionary
- Uses existing file if found
- Creates example if missing

**What you'll see:**
- File status (✓ found or 🔧 created)
- File location path
- Domain categories (common, business, educational)
- Total domain count

**Note:** Optional step - strategies can use default domains

In [ ]:
# Define domain dictionary path (in examples directory)
examples_dir = project_root / 'examples'
domain_path = examples_dir / 'data_examples' / 'email_domains.json'

# Check if file already exists
if domain_path.exists():
    print(f"✓ Found existing domain dictionary: {domain_path}")
    print(f"📂 Using existing file for domain configuration\n")
    
    # Load and display info
    with open(domain_path, 'r') as f:
        existing_domains = json.load(f)
    
    print("📊 Existing Dictionary Info:")
    print(f"  Format:   v{existing_domains.get('format_version', 'N/A')}")
    print(f"  Total Domains: {len(existing_domains.get('domains', []))}")
    
    print("\n💡 To use a different dictionary, replace or rename it.")
    print("=" * 80)

else:
    print(f"⚠️  Domain dictionary not found!")
    print(f"🔧 Creating example domain dictionary...\n")
    print("=" * 80)

    # Define domain structure
    domain_dict = {
        "format_version": "1.0",
        "description": "Custom email domain dictionary",
        "domains": [
            # Common domains
            "gmail.com", "yahoo.com", "hotmail.com", "outlook.com",
            "protonmail.com", "icloud.com", "aol.com",
            # Business domains
            "company.com", "enterprise.com", "business.com",
            "corporate.com", "organization.com", "firm.com",
            # Educational domains
            "university.edu", "college.edu", "school.edu",
            # Tech domains
            "tech.io", "startup.io", "dev.com"
        ]
    }
    
    # Save to file (with error handling)
    try:
        domain_path.parent.mkdir(parents=True, exist_ok=True)
        with open(domain_path, 'w') as f:
            json.dump(domain_dict, f, indent=2)
        print(f"✓ Example domain dictionary created: {domain_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {domain_path} (file may be open)")
        print(f"   Dictionary exists in memory only")

    print(f"\n📊 Statistics:")
    print(f"  Total domains: {len(domain_dict['domains'])}")
    print(f"  Domain categories:")
    print(f"    - Common: gmail.com, yahoo.com, hotmail.com, etc.")
    print(f"    - Business: company.com, enterprise.com, etc.")
    print(f"    - Educational: university.edu, college.edu, etc.")

    print("\n💡 You can edit this file to add your own domains!")
    print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for email generation
   - `"email_field"`: Column containing email addresses
   - `"first_name_field"`: Column with first names
   - `"last_name_field"`: Column with last names
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "email_field": "email",           # Field containing email addresses
    "first_name_field": "first_name", # Field with first names
    "last_name_field": "last_name"    # Field with last names
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_email_001",
    task_type="multi_strategy_email",
    description="Multi-strategy email generation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Name-Surname Format with Domain Preservation

**How to use:**
- Uses name.surname format (e.g., john.smith@domain.com)
- Preserves 70% of original domains
- Run to execute name-based strategy

**Key parameters:**
- `format='name_surname'`: First.Last pattern
- `preserve_domain_ratio=0.7`: Keep 70% original domains
- `separator_options=['.']`: Only dot separator
- `mode='ENRICH'`: Keeps original + adds generalized field

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Email examples with preserved domains

**Note:** Creates `email_name_surname` with realistic professional format

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: NAME-SURNAME FORMAT")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Name-Surname",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = FakeEmailOperation(
    # Core parameters
    field_name=FIELD_CONFIG["email_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['email_field']}_name_surname",
    output_format='csv',
    
    # Email format
    format='name_surname',
    first_name_field=FIELD_CONFIG["first_name_field"],
    last_name_field=FIELD_CONFIG["last_name_field"],
    
    # Domain configuration
    preserve_domain_ratio=0.7,  # Keep 70% original domains
    business_domain_ratio=0.2,
    
    # Format settings
    separator_options=['.'],  # Only dot separator
    number_suffix_probability=0.3,
    max_length=254,
    
    # Validation
    validate_source=True,
    handle_invalid_email='generate_new',
    
    # Consistency
    consistency_mechanism='mapping',
    key='strategy1-key',
    save_mapping=True,
    id_field='resume_id',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    detailed_metrics=True,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_name_surname',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_name_surname' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_email = FIELD_CONFIG["email_field"]
    output_field_s1 = f"{field_email}_name_surname"
    
    print(f"\n📧 Sample Generated Emails:")
    print(result_df_s1[[field_email, output_field_s1]].head(10))
    
    # Domain analysis
    orig_domains = result_df_s1[field_email].str.split('@').str[-1]
    gen_domains = result_df_s1[output_field_s1].str.split('@').str[-1]
    preserved = (orig_domains == gen_domains).sum()
    print(f"\n📊 Domain Preservation: {preserved}/{len(result_df_s1)} ({preserved/len(result_df_s1)*100:.1f}%)")

## STRATEGY 2: Surname-Name Format with Business Domains

**How to use:**
- Uses surname_name format (e.g., jsmith@business.com)
- Prioritizes business domains (80%)
- Run to execute surname based strategy

**Key parameters:**
- `format='surname_name'`: First + Last name
- `business_domain_ratio=0.8`: 80% business domains
- `preserve_domain_ratio=0.1`: Low original domain preservation
- `separator_options=['']`: No separator

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Compact email format examples

**Note:** Best for corporate environments with standard email patterns

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: SURNAME-NAME FORMAT")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Surname",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = FakeEmailOperation(
    field_name=FIELD_CONFIG["email_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['email_field']}_surname_name",
    output_format='csv',
    
    # Email format
    format='surname_name',
    first_name_field=FIELD_CONFIG["first_name_field"],
    last_name_field=FIELD_CONFIG["last_name_field"],
    
    # Domain configuration - prioritize business
    preserve_domain_ratio=0.1,  # Low preservation
    business_domain_ratio=0.8,  # High business domains
    
    # Format settings
    separator_options=[''],  # No separator
    number_suffix_probability=0.5,
    
    # Consistency
    consistency_mechanism='mapping',
    key='strategy2-key',
    save_mapping=True,
    id_field='resume_id',
    
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    detailed_metrics=True,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_surname_name',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_surname_name' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    output_field_s2 = f"{FIELD_CONFIG['email_field']}_surname_name"
    
    print(f"\n📧 Sample Generated Emails:")
    print(result_df_s2[[FIELD_CONFIG['email_field'], output_field_s2]].head(10))
    
    # Business domain analysis
    gen_domains = result_df_s2[output_field_s2].str.split('@').str[-1]
    business_domains = ['company.com', 'enterprise.com', 'business.com', 'corporate.com', 'organization.com', 'firm.com']
    business_count = gen_domains.isin(business_domains).sum()
    print(f"\n📊 Business Domains: {business_count}/{len(result_df_s2)} ({business_count/len(result_df_s2)*100:.1f}%)")

## STRATEGY 3: Nickname Format with Random Domains

**How to use:**
- Uses nickname format for casual/personal emails
- Random domain selection (no preservation)
- Run to execute nickname-based strategy

**Key parameters:**
- `format='nickname'`: Generate nickname-style emails
- `preserve_domain_ratio=0.0`: No domain preservation
- `business_domain_ratio=0.1`: Mostly common domains
- `separator_options=['_', '.', '']`: Mixed separators

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Casual email format examples

**Note:** Optimal for personal/informal email contexts with maximum diversity

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: NICKNAME FORMAT")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Nickname",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = FakeEmailOperation(
    field_name=FIELD_CONFIG["email_field"],
    mode='ENRICH',
    output_field_name=f"{FIELD_CONFIG['email_field']}_nickname",
    output_format='csv',
    
    # Email format
    format='nickname',
    first_name_field=FIELD_CONFIG["first_name_field"],
    last_name_field=FIELD_CONFIG["last_name_field"],
    
    # Domain configuration - random
    preserve_domain_ratio=0.0,  # No preservation
    business_domain_ratio=0.1,  # Mostly common domains
    
    # Format settings - more variety
    separator_options=['_', '.', ''],
    number_suffix_probability=0.6,
    
    # Consistency
    consistency_mechanism='mapping',
    key='strategy3-key',
    save_mapping=True,
    id_field='resume_id',
    
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    detailed_metrics=True,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_nickname',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_nickname' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    output_field_s3 = f"{FIELD_CONFIG['email_field']}_nickname"
    
    print(f"\n📧 Sample Generated Emails:")
    print(result_df_s3[[FIELD_CONFIG['email_field'], output_field_s3]].head(10))
    
    # Domain diversity analysis
    gen_domains = result_df_s3[output_field_s3].str.split('@').str[-1]
    domain_diversity = gen_domains.nunique()
    print(f"\n📊 Domain Diversity: {domain_diversity} unique domains")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and format characteristics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Email format comparison
- Domain preservation rates

**Note:** Consider both speed and business requirements when selecting strategy

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print("\n📧 Format Characteristics:")
print("  Strategy 1: name.surname@domain (Professional)")
print("  Strategy 2: jsurname@domain (Corporate)")
print("  Strategy 3: nickname123@domain (Casual)")

print("\n🌐 Domain Strategy:")
print("  Strategy 1: 70% domain preservation")
print("  Strategy 2: 80% business domains")
print("  Strategy 3: Random distribution")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Displays visualizations inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Generation performance, domain stats, format distribution
2. **Email Mapping**: Original → Synthetic email mappings
3. **Visualizations**: Domain and format charts (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_name_surname', 'Strategy 1: Name-Surname'),
    ('strategy2_surname_name', 'Strategy 2: Surname-Name'),
    ('strategy3_nickname', 'Strategy 3: Nickname')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Performance
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"   Generation Time: {perf.get('generation_time', 0):.4f}s")
                    print(f"   Records/sec: {perf.get('records_per_second', 0):,}")
                
                # Email generator metrics
                if 'email_generator' in metrics:
                    eg = metrics['email_generator']
                    print(f"   Format: {eg.get('format')}")
                    
                    # Domain distribution
                    dd = eg.get('domain_distribution', {})
                    if dd:
                        print(f"   Unique Domains: {dd.get('unique_domains', 0)}")
                        print(f"   Diversity Ratio: {dd.get('diversity_ratio', 0):.4f}")
                        
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Mapping (NEWEST)
    mapping_files = sorted(
        list(strategy_dir.glob('**/*mapping*.json')),
        key=lambda x: x.stat().st_mtime, reverse=True
    )
    if mapping_files:
        print(f"\n📄 MAPPING: {mapping_files[0].name}")
        try:
            with open(mapping_files[0], 'r') as f:
                mapping_data = json.load(f)
                mappings = mapping_data.get('mappings', {})
                if FIELD_CONFIG['email_field'] in mappings:
                    field_mappings = mappings[FIELD_CONFIG['email_field']]
                    print(f"   Total mappings: {len(field_mappings)}")
                    print(f"   Sample (first 3):")
                    for mapping in field_mappings[:3]:
                        print(f"     {mapping.get('original', 'N/A')} → {mapping.get('synthetic', 'N/A')}")
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Privacy Metrics

**How to use:**
- Calculate k-anonymity for original and synthetic emails
- Compare privacy improvements across strategies
- Run AFTER all 3 strategies complete

**What you'll see:**
- Original data: min k, avg k, vulnerable group count
- After anonymization: improved k-values with multipliers
- Email uniqueness preservation metrics

**Privacy targets:**
- Min k ≥ 5: Basic privacy protection
- Min k ≥ 10: Strong privacy protection
- Email diversity: Maintain reasonable uniqueness

**Note:** Email fields combined with other quasi-identifiers for k-anonymity

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics."""
    existing_qis = [q for q in quasi_identifiers if q in df.columns]
    if not existing_qis:
        return None

    group_sizes = df.groupby(existing_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

# Check if strategies completed
try:
    field_email = FIELD_CONFIG["email_field"]
    field_spec= "specialization"
    
    # Original
    k_orig = calculate_k_anonymity(df, [field_email, field_spec])
    print(f"📊 ORIGINAL DATA:")
    print(f"   Min k: {k_orig['min_k']}")
    print(f"   Avg k: {k_orig['avg_k']:.2f}")
    print(f"   Vulnerable groups: {k_orig['vulnerable_groups']}")
    
    # Email uniqueness
    print(f"\n📧 EMAIL UNIQUENESS:")
    print(f"   Original: {df[field_email].nunique()} unique emails")
    
    # After each strategy
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        output_s1 = f"{field_email}_name_surname"
        quasi_ids_s1 = [output_s1, field_spec]
        k_s1 = calculate_k_anonymity(result_df_s1, quasi_ids_s1)
        
        print(f"\n✨ STRATEGY 1 (Name-Surname):")
        print(f"   Min k: {k_s1['min_k']} ({k_s1['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Unique emails: {result_df_s1[output_s1].nunique()}")
    
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        output_s2 = f"{field_email}_surname_name"
        quasi_ids_s2 = [output_s2, field_spec]
        k_s2 = calculate_k_anonymity(result_df_s2, quasi_ids_s2)
        
        print(f"\n✨ STRATEGY 2 (Surname-Name):")
        print(f"   Min k: {k_s2['min_k']} ({k_s2['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Unique emails: {result_df_s2[output_s2].nunique()}")
    
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        output_s3 = f"{field_email}_nickname"
        quasi_ids_s3 = [output_s3, field_spec]
        k_s3 = calculate_k_anonymity(result_df_s3, quasi_ids_s3)
        
        print(f"\n✨ STRATEGY 3 (Nickname):")
        print(f"   Min k: {k_s3['min_k']} ({k_s3['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Unique emails: {result_df_s3[output_s3].nunique()}")
        
except NameError:
    print("⚠️  FIELD_CONFIG not defined!")
    print("   Please run 'Step 4: Setup Shared Environment' first")

## Step 8: Export Final Data

**How to use:**
- Export final synthetic emails from all strategies
- Each strategy gets its own output folder
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Available columns list
- Export confirmation (file path, row count)
- Preview of first 5 rows
- Domain distribution statistics

**Output structure:**
```
advanced_output/
├── strategy1_name_surname/email_name_surname_data.csv
├── strategy2_surname_name/email_surname_name_data.csv
└── strategy3_nickname/email_nickname_data.csv
```

**Note:** Files include both original and synthetic emails for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_email = FIELD_CONFIG["email_field"]
    field_first = FIELD_CONFIG["first_name_field"]
    field_last = FIELD_CONFIG["last_name_field"]
except NameError:
    print("❌ FIELD_CONFIG not defined! Run Step 4 first.")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Name-Surname
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: NAME-SURNAME")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_name_surname'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_email}_name_surname"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['employee_id', field_first, field_last, 
                          field_email, output_col_s1, field_spec]
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        output_path_s1 = s1_dir / 'email_name_surname_data.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
        
        # Domain stats
        domains = final_df_s1[output_col_s1].str.split('@').str[-1]
        print(f"\n📈 Domain Distribution:")
        print(f"   Unique domains: {domains.nunique()}")
        print(f"   Top 3 domains:")
        for domain, count in domains.value_counts().head(3).items():
            print(f"     {domain}: {count} ({count/len(domains)*100:.1f}%)")
    else:
        print(f"\n❌ Column '{output_col_s1}' not found")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Surname-Name
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: SURNAME-NAME")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_surname_name'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_email}_name_surname"
    output_col_s2 = f"{field_email}_surname_name"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['employee_id', field_first, field_last,
                          field_email, output_col_s1, output_col_s2, field_spec]
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'email_surname_name_data.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
        
        # Business domain stats
        domains = final_df_s2[output_col_s2].str.split('@').str[-1]
        business_domains = ['company.com', 'enterprise.com', 'business.com']
        business_count = domains.isin(business_domains).sum()
        print(f"\n📈 Business Domains: {business_count} ({business_count/len(domains)*100:.1f}%)")
    else:
        print(f"\n❌ Column '{output_col_s2}' not found")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Nickname
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: NICKNAME")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_nickname'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_email}_name_surname"
    output_col_s2 = f"{field_email}_surname_name"
    output_col_s3 = f"{field_email}_nickname"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['employee_id', field_first, field_last,
                          field_email, output_col_s1, output_col_s2, 
                          output_col_s3, field_spec]
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'email_nickname_data.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
        
        # Diversity stats
        domains = final_df_s3[output_col_s3].str.split('@').str[-1]
        print(f"\n📈 Domain Diversity: {domains.nunique()} unique domains")
    else:
        print(f"\n❌ Column '{output_col_s3}' not found")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 email generation strategies implemented and compared
- ✅ Privacy metrics calculated (k-anonymity, uniqueness)
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Name-based formats preserve professional appearance
- Surname-based formats provide corporate standardization
- Nickname formats maximize diversity and casualness
- Domain preservation balances utility and privacy
- Mapping consistency ensures deterministic generation

**Strategy recommendations:**
- **Use Strategy 1** when: Professional appearance matters, need domain preservation
- **Use Strategy 2** when: Corporate standardization required, business domains preferred
- **Use Strategy 3** when: Maximum privacy needed, personal/casual context

**Next steps:**
- Test with your own datasets
- Customize domain dictionaries
- Adjust format ratios for mixed patterns
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)